# 09 - Temporal Lead-Lag Analysis

This notebook investigates whether public or media sentiment leads or lags the other over the 2024 time window.

Steps:
1. Construct weekly sentiment time series for public and media
2. Stationarity check (ADF test); first-difference if needed
3. Cross-correlation at lags -8 to +8 weeks
4. Granger causality test (both directions)
5. Volume correlation analysis

**Caveat:** ~44 weekly observations is marginal for Granger causality. Inconclusive results are a valid finding.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import json

sys.path.append(str(Path('..').resolve()))
from analysis_utils import ensure_sentiment_columns, parse_date_safe, standardize_date_column

DATA_DIR = '../data'
FIG_DIR = '../figures'

df_public = pd.read_csv(f'{DATA_DIR}/public_with_sentiment.csv')
df_media = pd.read_csv(f'{DATA_DIR}/media_with_sentiment.csv')

df_public = ensure_sentiment_columns(df_public)
df_media = ensure_sentiment_columns(df_media)
df_public = standardize_date_column(df_public, 'date')
df_media = standardize_date_column(df_media, 'date')

df_public['date'] = parse_date_safe(df_public['date'])
df_media['date'] = parse_date_safe(df_media['date'])

print(f'Public: {len(df_public)} docs, date range: {df_public["date"].min()} to {df_public["date"].max()}')
print(f'Media: {len(df_media)} docs, date range: {df_media["date"].min()} to {df_media["date"].max()}')


In [ ]:
# Weekly aggregation
df_public['week'] = df_public['date'].dt.to_period('W')
df_media['week'] = df_media['date'].dt.to_period('W')

pub_weekly = df_public.groupby('week').agg(
    sentiment=('sentiment', 'mean'), count=('sentiment', 'size')).reset_index()
med_weekly = df_media.groupby('week').agg(
    sentiment=('sentiment', 'mean'), count=('sentiment', 'size')).reset_index()

pub_weekly['week_str'] = pub_weekly['week'].astype(str)
med_weekly['week_str'] = med_weekly['week'].astype(str)
weekly = pub_weekly.merge(med_weekly, on='week_str', suffixes=('_pub', '_med'))

print(f'Common weeks: {len(weekly)}')
print(f'Public weekly mean docs: {weekly["count_pub"].mean():.1f}')
print(f'Media weekly mean docs: {weekly["count_med"].mean():.1f}')

In [ ]:
# Figure: Weekly sentiment time series
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(weekly)), weekly['sentiment_pub'].values,
        'o-', color='#2196F3', linewidth=1.5, markersize=4, label='Public')
ax.plot(range(len(weekly)), weekly['sentiment_med'].values,
        's-', color='#FF9800', linewidth=1.5, markersize=4, label='Media')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Week Index')
ax.set_ylabel('Mean VADER Sentiment')
ax.set_title('Weekly Sentiment: Public vs Media', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/weekly_sentiment_timeseries.png', dpi=150)
plt.show()

In [ ]:
# Stationarity tests (ADF)
pub_ts = weekly['sentiment_pub'].values
med_ts = weekly['sentiment_med'].values

adf_pub = adfuller(pub_ts, maxlag=4)
adf_med = adfuller(med_ts, maxlag=4)
print(f'ADF Public:  stat={adf_pub[0]:.3f}, p={adf_pub[1]:.4f}')
print(f'ADF Media:   stat={adf_med[0]:.3f}, p={adf_med[1]:.4f}')

use_diff = False
if adf_pub[1] > 0.05 or adf_med[1] > 0.05:
    print('Non-stationary detected, applying first differencing')
    pub_ts_use = np.diff(pub_ts)
    med_ts_use = np.diff(med_ts)
    use_diff = True
else:
    pub_ts_use = pub_ts
    med_ts_use = med_ts

In [ ]:
# Cross-correlation
max_lag = min(8, len(pub_ts_use) // 3)
cc_lags = range(-max_lag, max_lag + 1)
cc_values = []
for lag in cc_lags:
    if lag < 0:
        cc = np.corrcoef(pub_ts_use[:lag], med_ts_use[-lag:])[0, 1]
    elif lag > 0:
        cc = np.corrcoef(pub_ts_use[lag:], med_ts_use[:-lag])[0, 1]
    else:
        cc = np.corrcoef(pub_ts_use, med_ts_use)[0, 1]
    cc_values.append(cc if not np.isnan(cc) else 0)

best_idx = np.argmax(np.abs(cc_values))
best_lag = list(cc_lags)[best_idx]
best_cc = cc_values[best_idx]
print(f'Peak cross-correlation: lag={best_lag} weeks, r={best_cc:.3f}')

# Figure
diff_label = ' (first-differenced)' if use_diff else ''
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(list(cc_lags), cc_values, color='#2196F3', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5)
ci = 1.96 / np.sqrt(len(pub_ts_use))
ax.axhline(ci, color='red', linestyle='--', alpha=0.5, label=f'95% CI ({ci:.2f})')
ax.axhline(-ci, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel(f'Lag (weeks){diff_label}')
ax.set_ylabel('Cross-Correlation')
ax.set_title('Cross-Correlation: Public vs Media Sentiment', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/crosscorrelation.png', dpi=150)
plt.show()

In [ ]:
# Granger causality tests
granger_max = min(4, len(pub_ts_use) // 5)
print(f'Testing with max lag = {granger_max}')
print(f'Caveat: ~{len(weekly)} weekly observations is marginal for Granger causality\n')

# Media -> Public
print('Media -> Public (does media Granger-cause public sentiment?):')
data_m2p = np.column_stack([pub_ts_use, med_ts_use])
gc_m2p = grangercausalitytests(data_m2p, maxlag=granger_max, verbose=True)

print('\nPublic -> Media (does public Granger-cause media sentiment?):')
data_p2m = np.column_stack([med_ts_use, pub_ts_use])
gc_p2m = grangercausalitytests(data_p2m, maxlag=granger_max, verbose=True)

In [ ]:
# Volume correlation
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(np.arange(len(weekly)) - 0.2, weekly['count_pub'].values,
       0.4, label='Public', color='#2196F3', alpha=0.7)
ax.bar(np.arange(len(weekly)) + 0.2, weekly['count_med'].values,
       0.4, label='Media', color='#FF9800', alpha=0.7)
ax.set_xlabel('Week Index')
ax.set_ylabel('Document Count')
ax.set_title('Weekly Volume: Public vs Media', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

vol_corr = np.corrcoef(weekly['count_pub'].values, weekly['count_med'].values)[0, 1]
print(f'Volume correlation: r={vol_corr:.3f}')